# Rank Log Diff Only

Confronto minimale tra baseline semplice e LightGBM usando una sola feature di ranking.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

import lightgbm as lgb


In [ ]:
PROCESSED_DIR = Path("../data/processed")

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")


In [ ]:
FEATURE_COLS = ["rank_log_diff"]
TARGET_COL = "target"


In [ ]:
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL]

X_test = test_df[FEATURE_COLS].copy()
y_test = test_df[TARGET_COL]

for col in FEATURE_COLS:
    fill_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)


In [ ]:
model = lgb.LGBMClassifier(
    objective="binary",
    metric="binary_logloss",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    num_leaves=7,
    min_child_samples=30,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42
)


In [ ]:
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(50)
    ]
)


In [ ]:
test_pred_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, test_pred)
ll = log_loss(y_test, test_pred_proba)
auc = roc_auc_score(y_test, test_pred_proba)

acc, ll, auc


In [ ]:
baseline_score = test_df["rank_log_diff"]
baseline_proba = 1 / (1 + np.exp(-baseline_score))
baseline_pred = (baseline_proba >= 0.5).astype(int)

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_ll = log_loss(y_test, baseline_proba)
baseline_auc = roc_auc_score(y_test, baseline_proba)

baseline_acc, baseline_ll, baseline_auc


In [ ]:
pd.DataFrame({
    "metric": ["accuracy", "log_loss", "auc"],
    "simple_baseline": [baseline_acc, baseline_ll, baseline_auc],
    "lgbm_model": [acc, ll, auc],
})
